# Assignment 5 - Pandas Data Analysis
**Student:** Kanshar Maksotov  
**Group:** Po 3-23

---
## Section 1 — Load & profile

Loaded both files and checked shape, types, missing values, duplicates, and country values.

In [ ]:
import pandas as pd

sales = pd.read_csv('sales_messy.csv')
customers = pd.read_csv('customers.csv')

print("=== sales_messy.csv ===")
print("Shape:", sales.shape)


In [ ]:
sales.info()


In [ ]:
print("Missing values per column:")
print(sales.isnull().sum())


In [ ]:
print("Duplicate rows:", sales.duplicated().sum())


In [ ]:
print("country.unique():")
print(sales['country'].unique())


**Found Problems:**

1. **Missing values:** `customer_id` — 8 rows, `unit_price` — 13 rows, `discount` — 20 rows.
2. **Duplicate rows:** 8 exact duplicates that inflate counts.
3. **Inconsistent country values:** `'GERMANY'` vs `'Germany'`, `' France'` (leading space), `' kazakhstan '` (both spaces and wrong case), `'uk '` (trailing space) — same country stored multiple ways.
4. **`order_date`** is dtype `str` (object), not `datetime` — can't use for time analysis yet.
5. **`customer_id`** is `float64` due to NaN rows — should be integer after we drop nulls.


---
## Section 2 — Clean

Cleaned the data step by step — removed duplicates, fixed country values, filled missing fields, and parsed dates.
After cleaning, confirmed zero missing values in the key columns.


In [ ]:
# remove duplicates
df = sales.drop_duplicates()
print("After drop_duplicates:", df.shape)


In [ ]:
# standardise country
df['country'] = df['country'].str.strip().str.title()
print("Unique countries after standardisation:", df['country'].unique())


In [ ]:
# fill missing discount (with 0)
df['discount'] = df['discount'].fillna(0)

# fill unit_price with median
price_median = df['unit_price'].median()
print(f"Median unit_price used for fill: {price_median}")
df['unit_price'] = df['unit_price'].fillna(price_median)


In [ ]:
# drop rows with no customer_id
df = df.dropna(subset=['customer_id'])
df['customer_id'] = df['customer_id'].astype(int)
print("Shape after dropping null customer_id rows:", df.shape)


In [ ]:
# parse order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'])
print("order_date dtype:", df['order_date'].dtype)


In [ ]:
# ── Proof: zero missing values in cleaned columns ──
target_cols = ['customer_id', 'order_date', 'unit_price', 'discount']
missing_after = df[target_cols].isnull().sum()
print("Missing values after cleaning:")
print(missing_after)
assert missing_after.sum() == 0, "Still have missing values — check cleaning steps!"
print("✓ All zeros — cleaning complete.")


---
## Section 3 — Enrich

Added a revenue column and a month column for further analysis.


In [ ]:
df['revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount'])
df['month']   = df['order_date'].dt.to_period('M')

print("New columns added:")
print(df[['order_date', 'month', 'quantity', 'unit_price', 'discount', 'revenue']].head(6))


---
## Section 4 — Merge

Joined customer data to get segment and name. Checked that row count stayed the same after the merge.


In [ ]:
rows_before = len(df)

merged = df.merge(
    customers[['customer_id', 'customer_name', 'segment']],
    on='customer_id',
    how='left'
)

rows_after = len(merged)
print(f"Rows before merge: {rows_before}")
print(f"Rows after  merge: {rows_after}")
assert rows_before == rows_after, "Row count changed — merge created duplicates or dropped rows!"
print("✓ Row count unchanged.")


In [ ]:
print(merged[['order_id', 'customer_id', 'customer_name', 'segment', 'revenue']].head(5))


---
## Section 5 — Aggregate

Answered three business questions — revenue by category, by month, and by customer segment.


### 1 — Revenue per Category


In [ ]:
total_revenue = merged['revenue'].sum()

rev_cat = (
    merged.groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
)
rev_cat['share_%'] = (rev_cat['total_revenue'] / total_revenue * 100).round(2)
rev_cat['total_revenue'] = rev_cat['total_revenue'].round(2)
print(rev_cat.to_string(index=False))


### 2 — Revenue per Month


In [ ]:
rev_month = (
    merged.groupby('month')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
)
rev_month['total_revenue'] = rev_month['total_revenue'].round(2)
print(rev_month.to_string(index=False))


### 3 — Revenue per Customer Segment


In [ ]:
rev_seg = (
    merged.groupby('segment')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
)
rev_seg['share_%'] = (rev_seg['total_revenue'] / total_revenue * 100).round(2)
rev_seg['total_revenue'] = rev_seg['total_revenue'].round(2)
print(rev_seg.to_string(index=False))


---
## Section 6 — Conclusions

Key findings from the analysis, each backed by a number from the tables above.

---

**Finding 1 — Laptops dominate revenue.**  
The Laptops category generated **161 187.40** in revenue, accounting for **54.97 %** of total revenue (293 209.90). The next category, Phones, earned 63 403.40 — less than half of Laptops. This concentration means the business is heavily dependent on one product line.

**Finding 2 — July 2025 was the strongest month.**  
Monthly revenue peaked in July 2025 at **42 529.43**, roughly 1.4× the second-best month (October: 33 697.75). A seasonal or campaign-driven spike in mid-year is worth investigating.

**Finding 3 — Consumer segment drives most revenue.**  
Consumer customers produced **174 850.84** (59.6 % of total), compared with Education (77 782.03, 26.5 %) and Business (40 577.03, 13.8 %). Despite typically having smaller budgets, individual consumers collectively outspend business accounts.

**Finding 4 — Accessories are a very minor revenue source.**  
Accessories contributed only **10 323.55** — just 3.52 % of total revenue — despite likely having many transactions. This suggests either very low unit prices, small order sizes, or both. Accessories may serve mainly as add-ons rather than a primary revenue driver.

**Finding 5 — 8 rows (3.9 % of raw data) were lost to data quality issues.**  
We removed 8 duplicates and 8 rows with no `customer_id` (16 rows, ~7.7 %), leaving 193 clean rows from the original 208. The impact on aggregate revenue is hard to quantify because missing `customer_id` rows cannot be attributed to a segment — a reminder that upstream data quality directly affects analysis completeness.
